In [20]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

## PIPELINE

El Pipeline convierte age y gender en variables binarias (OneHot), normaliza todas las variables numéricas (StandardScaler) y entrena un modelo de clasificación automáticamente. 

In [21]:
X = pd.read_csv("../data/clean/X_clean.csv")
y = pd.read_csv("../data/clean/y.csv").squeeze()

añadimos categoricas

In [22]:
cat_cols = ["age", "gender", "merchant", "customer","category"]
num_cols = X.drop(columns=cat_cols).columns.tolist()

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )
}

In [26]:
results = {}

for name, model in models.items():
    
    clf = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])
    
    clf.fit(X_train, y_train)
    
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_pred_proba)
    results[name] = auc
    que 
    print(f"{name} AUC: {auc:.4f}")

RandomForest AUC: 0.9933
XGBoost AUC: 0.9981
[LightGBM] [Info] Number of positive: 5760, number of negative: 469954
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007936 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8691
[LightGBM] [Info] Number of data points in the train set: 475714, number of used features: 4074
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.012108 -> initscore=-4.401697
[LightGBM] [Info] Start training from score -4.401697


/opt/miniconda3/envs/tfg_env/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM AUC: 0.9983


In [27]:
results_sorted = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))
print(results_sorted)

{'LightGBM': 0.9983233270774285, 'XGBoost': 0.9981191649383725, 'RandomForest': 0.9933406590782494}
